In [ ]:
# Per-class analysis of CellWhisperer evaluation results
# Compare trimodal vs bimodal model performance at the class level
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import numpy as np
from collections import defaultdict
import re

metric = snakemake.params.metric
print("Starting per-class CellWhisperer analysis...")

In [ ]:
# Load CellWhisperer evaluation results from performance_metrics_permetadata files
results = []

# Parse input files to extract model type, dataset, and metadata
for result_file in snakemake.input:
    df = pd.read_csv(result_file)

    # Extract model type, dataset, and metadata from file path
    path_parts = Path(result_file).parts

    # Find model name (spotwhisperer_*)
    model_name = None
    for part in path_parts:
        if part.startswith("spotwhisperer_") and part != "spotwhisperer_eval":
            model_name = part.replace("spotwhisperer_", "")
            break

    # Extract dataset and metadata column
    dataset_idx = None
    for i, part in enumerate(path_parts):
        if part == "datasets":
            dataset_idx = i + 1
            break

    if dataset_idx and dataset_idx < len(path_parts):
        dataset = path_parts[dataset_idx]
        metadata_col = (
            path_parts[dataset_idx + 1]
            if dataset_idx + 1 < len(path_parts)
            else "celltype"
        )
    else:
        continue

    # Determine model type based on dataset combination
    if model_name == "cellxgene_census__archs4_geo__hest1k__quilt1m":
        model_type = "trimodal"
    elif model_name == "hest1k__quilt1m":
        model_name = "bimodal_bridge"
    elif model_name == "cellxgene_census__archs4_geo":
        model_type = (
            "bimodal_matching"
            if dataset
            in [
                "tabula_sapiens",
                "tabula_sapiens_well_studied_celltypes",
                "pancreas",
                "human_disease",
                "immgen",
            ]
            else "bimodal_mismatch1"
        )
    elif model_name == "hest1k":
        model_type = "bimodal_mismatch1"
    elif model_name == "quilt1m":
        model_type = "bimodal_mismatch2"
    else:
        model_type = "unknown"

    # Add metadata to each row
    df["model_type"] = model_type
    df["model_name"] = model_name
    df["dataset"] = dataset
    df["metadata_col"] = metadata_col
    df["result_file"] = str(result_file)

    results.append(df)

combined_df = pd.concat(results, ignore_index=True)
print(f"Loaded {len(combined_df)} per-class metric entries from {len(results)} files")
print(combined_df.head())

In [ ]:
combined_df[[v for v in combined_df.columns if not v.startswith("n_sam")]]

In [ ]:
combined_df[[v for v in combined_df.columns if not v.startswith("n_sam")]]

In [ ]:
# Create comparison between trimodal and bimodal models
# Focus on {metric} score as the main metric

# Reset index to make sure we can access the class information
combined_df = combined_df.reset_index()

comparison_dfs = []

for dataset in combined_df["dataset"].unique():
    dataset_data = combined_df[combined_df["dataset"] == dataset].copy()

    # Check available metrics
    available_metrics = [
        col for col in dataset_data.columns if "{metric}" in col.lower()
    ]
    print(f"Available metrics for {dataset}: {available_metrics}")

    # Pivot to get model types as columns for each cell type
    pivot_df = dataset_data.pivot_table(
        index=["dataset", "class"],
        columns="model_type",
        values=snakemake.params.metric,
        aggfunc="mean",
    ).reset_index()

    # Calculate improvement (trimodal - bimodal_matching)
    if "trimodal" in pivot_df.columns and "bimodal_matching" in pivot_df.columns:
        pivot_df["improvement"] = pivot_df["trimodal"] - pivot_df["bimodal_matching"]
        pivot_df["relative_improvement"] = (
            pivot_df["improvement"] / pivot_df["bimodal_matching"] * 100
        )
        comparison_dfs.append(pivot_df)

model_comparison = pd.concat(comparison_dfs, ignore_index=True)
print("Model comparison created:")
model_comparison

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages

with PdfPages(snakemake.output.plot) as pdf:
    for dataset, model_comparison_dataset in model_comparison.groupby("dataset"):
        # Create visualization
        fig, axes = plt.subplots(2, 2, figsize=(20, 12))

        fig.suptitle(
            f"Dataset: {dataset} CellWhisperer Per-Class Performance: Trimodal vs Bimodal Models",
            fontsize=16,
        )

        # Plot 1: Top 10 improving classes
        top_improving = model_comparison_dataset.nlargest(10, "improvement")
        y_pos = np.arange(len(top_improving))
        axes[0, 0].barh(y_pos, top_improving["improvement"], color="green", alpha=0.7)
        axes[0, 0].set_yticks(y_pos)
        axes[0, 0].set_yticklabels(
            [f"{row['dataset']}: {row['class']}" for _, row in top_improving.iterrows()]
        )
        axes[0, 0].set_xlabel(f"{metric} Score Improvement")
        axes[0, 0].set_title(
            f"Top 10 Classes: {metric} Score Improvement\n(Trimodal - Bimodal)"
        )
        axes[0, 0].grid(True, alpha=0.3)

        # Plot 2: Top 10 declining classes
        top_declining = model_comparison_dataset.nsmallest(10, "improvement")
        y_pos = np.arange(len(top_declining))
        axes[0, 1].barh(y_pos, top_declining["improvement"], color="red", alpha=0.7)
        axes[0, 1].set_yticks(y_pos)
        axes[0, 1].set_yticklabels(
            [f"{row['dataset']}: {row['class']}" for _, row in top_declining.iterrows()]
        )
        axes[0, 1].set_xlabel(f"{metric} Score Change")
        axes[0, 1].set_title(
            f"Top 10 Classes: {metric} Score Decline\n(Trimodal - Bimodal)"
        )
        axes[0, 1].grid(True, alpha=0.3)

        # Plot 3: Performance comparison by dataset
        dataset_comparison = model_comparison_dataset.groupby("dataset")[
            ["trimodal", "bimodal_matching", "improvement"]
        ].mean()
        x = np.arange(len(dataset_comparison))
        width = 0.35
        axes[1, 0].bar(
            x - width / 2,
            dataset_comparison["bimodal_matching"],
            width,
            label="Bimodal",
            alpha=0.8,
        )
        axes[1, 0].bar(
            x + width / 2,
            dataset_comparison["trimodal"],
            width,
            label="Trimodal",
            alpha=0.8,
        )
        axes[1, 0].set_xlabel("Datasets")
        axes[1, 0].set_ylabel(f"Mean {metric} Score")
        axes[1, 0].set_title(f"Mean {metric} Score by Dataset")
        axes[1, 0].set_xticks(x)
        axes[1, 0].set_xticklabels(dataset_comparison.index, rotation=45, ha="right")
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)

        # Plot 4: Distribution of improvements
        axes[1, 1].hist(
            model_comparison_dataset["improvement"], bins=20, alpha=0.7, color="blue"
        )
        axes[1, 1].axvline(x=0, color="red", linestyle="--", alpha=0.8)
        axes[1, 1].set_xlabel(f"{metric} Score Improvement")
        axes[1, 1].set_ylabel("Number of Classes")
        axes[1, 1].set_title(f"Distribution of {metric} Score Improvements")
        axes[1, 1].grid(True, alpha=0.3)

        plt.tight_layout()
        # plt.show()
        pdf.savefig(fig)

In [ ]:
# Statistical analysis of improvements
print("=== CELLWHISPERER STATISTICAL SUMMARY ===")

improvements = model_comparison["improvement"].dropna()

print(f"\n{metric} Score Improvements:")
print(f"Mean improvement: {improvements.mean():.4f}")
print(f"Median improvement: {improvements.median():.4f}")
print(f"Std deviation: {improvements.std():.4f}")
print(
    f"Classes with positive improvement: {(improvements > 0).sum()}/{len(improvements)}"
)
print(f"Max improvement: {improvements.max():.4f}")
print(f"Min improvement: {improvements.min():.4f}")

# Dataset-specific analysis
for dataset in model_comparison["dataset"].unique():
    dataset_data = model_comparison[model_comparison["dataset"] == dataset]
    dataset_improvements = dataset_data["improvement"].dropna()

    positive_improvements = (dataset_improvements > 0).sum()

    print(f"\n{dataset.upper()} Dataset:")
    print(f"  Mean {metric} improvement: {dataset_improvements.mean():.4f}")
    print(f"  Classes improved: {positive_improvements}/{len(dataset_improvements)}")

    # Top improving classes
    top_improved = dataset_data.nlargest(3, "improvement")
    print(f"  Top improving classes:")
    for _, row in top_improved.iterrows():
        print(f'    {row["class"]}: +{row["improvement"]:.4f}')

# Save detailed results
model_comparison.to_csv(snakemake.output.analysis, index=False)
print(f"\nDetailed results saved to: {snakemake.output.analysis}")